In [2]:
import cv2 
img = cv2.imread("test.jpg")

In [3]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

In [4]:
# histogram : distribution of pixel intensity values in an image
hist = cv2.calcHist([gray], [0], None, [256], [0,256])
equalized = cv2.equalizeHist(gray)

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
result = clahe.apply(gray)

In [5]:
import cv2

img = cv2.imread("test.jpg")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

equalized = cv2.equalizeHist(gray)

clahe = cv2.createCLAHE(2.0, (8,8))
clahe_img = clahe.apply(gray)

# cv2.imshow("Original", gray)
# cv2.imshow("Equalized", equalized)
# cv2.imshow("CLAHE", clahe_img)

# cv2.waitKey(0)
# cv2.destroyAllWindows()

In [6]:
# Feature detection = distinct, unique point in an image
# Corners
# Edges intersections
# Texture points

In [7]:
# Harris corner detection
import numpy as np
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gray = np.float32(gray)
corners = cv2.cornerHarris(gray, 2, 3, 0.04)

img[corners > 0.01 * corners.max()] = [0,0,255]
# cv2.imshow("Corners", img)
# cv2.waitKey(0)
# cv2.destroyAllWindows()

Detects strong corner points
Marks them in red

In [8]:
# key points : important points in an image that can be used for matching, tracking, etc.
# orb : Oriented FAST and Rotated BRIEF

orb = cv2.ORB_create()

kp, des = orb.detectAndCompute(img, None)

img_kp = cv2.drawKeypoints(img, kp, None)

# cv2.imshow("Keypoints", img_kp)
# cv2.waitKey(0)
# cv2.destroyAllWindows()

In [ ]:
img1 = cv2.imread("test.jpg", 0)
img2 = cv2.imread("shapes.jpg", 0)

orb = cv2.ORB_create()

kp1, des1 = orb.detectAndCompute(img1, None)
kp2, des2 = orb.detectAndCompute(img2, None)

bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

matches = bf.match(des1, des2)

matches = sorted(matches, key=lambda x: x.distance)

img_match = cv2.drawMatches(img1, kp1, img2, kp2, matches[:20], None)

# cv2.imshow("Matches", img_match)
# cv2.waitKey(0)
# cv2.destroyAllWindows()

In [11]:
# Detect features
# Convert to descriptors
# Compare descriptors
# Match similar ones

Face detection


Haar Cascade : A pre-trained model used in OpenCV

In [15]:
# uses black and white patterns
# eyes - dark regions
# cheeks - light regions

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

import cv2


In [16]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    
    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x,y), (x+w,y+h), (0,255,0), 2)
    
    cv2.imshow("Live Face Detection", frame)
    
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [17]:
# Limitations of Haar Cascade

# Not very accurate
# Fails with:
    # Different angles
    # Poor lighting
    # Masks

Object detection 

In [21]:
# what and where 
# image -> ( person, Car, dog ) + bounding box

# deep learning based object detection
# neural networks trained on large datasets to detect and classify objects in images

# YOLO : You Only Look Once ( detects in single pass )
# divides images into grid and predicts object, location and confidence score for each grid cell

In [ ]:
import cv2
import numpy as np

# Load YOLO
net = cv2.dnn.readNet("yolov3.weights", "yolov3.cfg")

# Load class names
with open("coco.names", "r") as f:
    classes = f.read().splitlines()

# Get output layers
output_layers = net.getUnconnectedOutLayersNames()

# Start webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    h, w, _ = frame.shape

    # Create blob
    blob = cv2.dnn.blobFromImage(
        frame,
        1/255,
        (416, 416),
        (0, 0, 0),
        swapRB=True,
        crop=False
    )

    net.setInput(blob)
    outputs = net.forward(output_layers)

    boxes = []
    confidences = []
    class_ids = []

    # Process detections
    for output in outputs:
        for detection in output:
            scores = detection[5:]
            class_id = np.argmax(scores)
            confidence = scores[class_id]

            if confidence > 0.5:
                center_x = int(detection[0] * w)
                center_y = int(detection[1] * h)
                width = int(detection[2] * w)
                height = int(detection[3] * h)

                x = int(center_x - width / 2)
                y = int(center_y - height / 2)

                boxes.append([x, y, width, height])
                confidences.append(float(confidence))
                class_ids.append(class_id)

    # Apply Non-Max Suppression
    indexes = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)

    if len(indexes) > 0:
        for i in indexes.flatten():
            x, y, w_box, h_box = boxes[i]
            label = classes[class_ids[i]]
            conf = confidences[i]

            cv2.rectangle(frame, (x, y), (x + w_box, y + h_box), (0, 255, 0), 2)
            cv2.putText(frame, f"{label} {conf:.2f}",
                        (x, y - 10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.5,
                        (0, 255, 0),
                        2)

    cv2.imshow("YOLO Object Detection", frame)

    # Press ESC to exit
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [24]:
# object tracking : following an object across frames in a video
# Why Tracking is Important
# Reduces computation
# Real-time systems
# Smooth motion tracking

In [ ]:
import cv2

# Start webcam
cap = cv2.VideoCapture(0)

ret, frame = cap.read()
if not ret:
    print("Error: Cannot read from camera")
    exit()

# Create tracker
tracker = cv2.TrackerCSRT_create()

# Select ROI AFTER getting frame
bbox = cv2.selectROI("Select Object", frame, False)

# Initialize tracker
tracker.init(frame, bbox)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    success, bbox = tracker.update(frame)

    if success:
        x, y, w, h = map(int, bbox)
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)
    else:
        cv2.putText(frame, "Tracking Lost", (50,50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

    cv2.imshow("Tracking", frame)

    # Press ESC to exit
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()